In [ ]:

# This model predicts user proficiency level (A1–C2)
# based on placement test quiz performance

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder


In [ ]:
# MongoDB connection via .env
import os
from pathlib import Path
from pymongo import MongoClient
from dotenv import load_dotenv

# Load .env from common locations
for env_path in [Path.cwd()/'.env', Path.cwd().parent/'.env', Path.cwd().parent.parent/'.env']:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"Loaded .env from {env_path}")
        break
else:
    load_dotenv()
    print("Loaded .env via default lookup")

MONGO_URI = os.getenv('MONGO_URI')
MONGO_DBNAME = os.getenv('MONGO_DBNAME')

if not MONGO_URI:
    raise RuntimeError('MONGO_URI not set in environment. Please add it to your .env file.')

print(f"Connecting to MongoDB: {MONGO_URI}")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=10000)
client.admin.command('ping')

# Prefer default DB from URI, else use env var
try:
    db = client.get_default_database()
except Exception:
    db = None

if db is None:
    if not MONGO_DBNAME:
        raise RuntimeError('No default DB in URI and MONGO_DBNAME not set. Define MONGO_DBNAME in .env.')
    db = client[MONGO_DBNAME]

print(f"✓ Connected to Database: {db.name}")

In [ ]:
# ============================================
# LOAD DATA FROM MONGODB
# ============================================
# Query Exam collection for features + join with users for target

print("\n--- LOADING DATA FROM MONGODB ---")
client.admin.command('ping')
print("✓ MongoDB connection active")

# Query Exam collection (stores placement test results with features)
print(f"\nAvailable collections: {db.list_collection_names()}")

# Mongoose lowercases and pluralizes model names by default
# Model "Exam" → collection "exams"
exam_collection_names = ['exams', 'Exam', 'exam']
exams_data = None

for collection_name in exam_collection_names:
    if collection_name in db.list_collection_names():
        print(f"Found collection: '{collection_name}'")
        exams_data = list(db[collection_name].find({}))
        print(f"Documents in '{collection_name}': {len(exams_data)}")
        break

if exams_data is None:
    raise ValueError(f"Exam collection not found. Available collections: {db.list_collection_names()}")

if len(exams_data) == 0:
    raise ValueError("No data found in exam collection. Please ensure placement test data exists.")

if len(exams_data) > 0:
    print(f"Sample exam record: {exams_data[0]}")

# Convert to DataFrame
df = pd.DataFrame(exams_data)
print(f"\nExam data shape: {df.shape}")
print(f"Exam columns: {df.columns.tolist()}")

# Drop MongoDB _id
if '_id' in df.columns:
    df = df.drop('_id', axis=1)

# Rename average_time to avg_time_per_question if it exists
if 'average_time' in df.columns:
    df = df.rename(columns={'average_time': 'avg_time_per_question'})
    print("Renamed 'average_time' → 'avg_time_per_question'")

# Load users and get expertise_lvl
print(f"\nLoading users from 'users' collection...")
users_data = list(db['users'].find({}, {'_id': 1, 'expertise_lvl': 1}))
users_df = pd.DataFrame(users_data)

if users_df.empty:
    raise ValueError("No users found in 'users' collection.")

# Rename _id to userID for merge
users_df = users_df.rename(columns={'_id': 'userID'})
print(f"Users loaded: {len(users_df)}")

# Merge exam features with user expertise level
df = df.merge(users_df, left_on='userID', right_on='userID', how='inner')
print(f"\nMerged dataset shape: {df.shape}")
print(f"Columns after merge: {df.columns.tolist()}")

# Remove records with missing target
df = df.dropna(subset=['expertise_lvl'])
print(f"Records after removing missing expertise_lvl: {len(df)}")

print(f"\nDataset info:")
print(f"Shape: {df.shape}")
print(f"Data types:\n{df.dtypes}")

# ============================================
# FEATURE SELECTION
# ============================================
FEATURES = [
    "total_questions",
    "correct_answers",
    "accuracy",
    "easy_accuracy",
    "medium_accuracy",
    "hard_accuracy",
    "avg_time_per_question"
]
TARGET = "expertise_lvl"  # From users schema (0-5)

# Verify all features exist
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"\n⚠ Warning: Missing features {missing}. Using available features only.")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nFinal features: {FEATURES}")
print(f"Target: {TARGET}")

X = df[FEATURES]
y = df[TARGET]

print(f"\nTarget (expertise_lvl) distribution:")
print(y.value_counts().sort_index())
print(f"\nFeature statistics:\n{X.describe()}")

In [ ]:
# ============================================
# TRAIN CATBOOST MODEL
# ============================================

import pickle
import datetime

# Check dataset size
print(f"Dataset size: {len(df)} records")
print(f"Unique target classes: {y.nunique()}")

if len(df) < 10:
    raise ValueError(f"⚠ Insufficient data: {len(df)} records. Need at least 10 records to train a model.")

if y.nunique() < 2:
    raise ValueError(f"⚠ No class variation: All samples have expertise_lvl={y.iloc[0]}. Need multiple classes.")

# Label encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Mapping:", label_mapping)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Model definition
model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=600,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=100
)

# Train
print("\n--- TRAINING MODEL ---")
model.fit(X_train, y_train)

# Evaluate
print("\n--- EVALUATION ---")
y_pred = model.predict(X_test)

f1 = f1_score(y_test, y_pred, average="macro")
print(f"\nMacro F1 Score: {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=[str(c) for c in label_encoder.classes_]))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save model to MongoDB
print("\n--- SAVING MODEL TO MONGODB ---")
model_bytes = pickle.dumps(model)

model_record = {
    "model_name": "expertise_level_classifier",
    "model_type": "CatBoost",
    "timestamp": datetime.datetime.now(),
    "features": FEATURES,
    "target_classes": label_encoder.classes_.tolist(),
    "macro_f1_score": float(f1),
    "model_binary": model_bytes,
    "hyperparameters": {
        "iterations": 600,
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5
    }
}

if 'db' in globals() and db is not None:
    ml_models = db.ml_models
    result = ml_models.update_one(
        {"model_name": "expertise_level_classifier"},
        {"$set": model_record},
        upsert=True
    )
    print(f"✓ Model saved to MongoDB")
    print(f"  - Modified: {result.modified_count}")
    print(f"  - Upserted ID: {result.upserted_id}")
else:
    print("⚠ MongoDB not available. Model not saved.")

In [ ]:
# ============================================
# PREDICTION API FUNCTION
# ============================================

def predict_expertise_level(user_features: dict):
    """
    Predict user expertise level (0-5) from exam features
    
    Args:
        user_features: Dict with keys from FEATURES list:
                      - total_questions (int)
                      - correct_answers (int)
                      - accuracy (float: 0-1)
                      - easy_accuracy (float: 0-1)
                      - medium_accuracy (float: 0-1)
                      - hard_accuracy (float: 0-1)
                      - avg_time_per_question (float: seconds)
    
    Returns:
        Dict with {expertise_level: 0-5, confidence: float}
    """
    try:
        # Load model
        ml_models = db.ml_models
        model_record = ml_models.find_one({"model_name": "expertise_level_classifier"})
        
        if not model_record:
            return {"error": "Model not found in MongoDB", "expertise_level": None}
        
        model = pickle.loads(model_record["model_binary"])
        features = model_record["features"]
        classes = model_record["target_classes"]
        
        # Prepare input
        input_df = pd.DataFrame([user_features])[features]
        
        # Predict
        encoded_pred = model.predict(input_df)[0]
        expertise_level = classes[int(encoded_pred)]
        
        # Get confidence
        probabilities = model.predict_proba(input_df)[0]
        confidence = float(max(probabilities))
        
        return {
            "expertise_level": int(expertise_level),
            "confidence": confidence,
            "probabilities": {str(c): float(p) for c, p in zip(classes, probabilities)}
        }
    
    except Exception as e:
        return {"error": str(e), "expertise_level": None}


In [ ]:
# ============================================
# LOAD MODEL FROM MONGODB
# ============================================

def load_model_from_mongo():
    """Load trained expertise level model from MongoDB"""
    try:
        ml_models = db.ml_models
        model_record = ml_models.find_one({"model_name": "expertise_level_classifier"})
        
        if not model_record:
            print("ERROR: Model not found. Train the model first.")
            return None, None, None
        
        model = pickle.loads(model_record["model_binary"])
        features = model_record["features"]
        classes = model_record["target_classes"]
        
        print(f"✓ Loaded model from MongoDB")
        print(f"  - Trained: {model_record['timestamp']}")
        print(f"  - F1 Score: {model_record['macro_f1_score']:.4f}")
        print(f"  - Features: {len(features)}")
        print(f"  - Classes: {classes}")
        
        return model, features, classes
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None, None

# Test loading
loaded_model, loaded_features, loaded_classes = load_model_from_mongo()


In [ ]:
# ============================================
# TEST WITH REAL DATABASE DATA
# ============================================
print("\n--- TESTING WITH REAL EXAM DATA ---")

# Fetch actual exam data from database (first record)
exam_collection_names = ['exams', 'Exam', 'exam']
exam_data = None

for collection_name in exam_collection_names:
    if collection_name in db.list_collection_names():
        exam_data = db[collection_name].find_one({})
        break

if not exam_data:
    raise ValueError(f"No exam data found in database. Available collections: {db.list_collection_names()}")

print(f"✓ Found real exam data from database")
print(f"  Record: {exam_data}")

# Prepare features (remove MongoDB _id and userID for prediction)
real_features = {k: v for k, v in exam_data.items() if k not in ['_id', 'userID', 'createdDate']}

# Rename average_time if needed
if 'average_time' in real_features:
    real_features['avg_time_per_question'] = real_features.pop('average_time')

# Make prediction
result = predict_expertise_level(real_features)
print(f"\n✓ Prediction from real exam data:")
print(f"  Expertise Level: {result.get('expertise_level')}")
print(f"  Confidence: {result.get('confidence', 0):.2%}")
if 'probabilities' in result:
    print(f"  Class Probabilities:")
    for cls, prob in sorted(result['probabilities'].items()):
        print(f"    Level {cls}: {prob:.4f}")
if 'error' in result:
    print(f"  Error: {result['error']}")


In [ ]:
# ============================================
# COMPREHENSIVE MODEL TESTING
# ============================================

print("\n" + "="*60)
print("MODEL TESTING SUITE")
print("="*60)

# Test 1: Model Loading Test
print("\n[TEST 1] Model Loading from MongoDB")
print("-" * 60)
loaded_model, loaded_features, loaded_classes = load_model_from_mongo()

if loaded_model is not None:
    print("✓ Model loaded successfully")
else:
    print("✗ Model loading failed")

# Test 2: Prediction on Multiple Samples
print("\n[TEST 2] Predictions on Multiple Exam Records")
print("-" * 60)

# Get multiple exam records for testing
for collection_name in exam_collection_names:
    if collection_name in db.list_collection_names():
        test_exams = list(db[collection_name].find({}).limit(5))
        break

if test_exams:
    print(f"Testing on {len(test_exams)} exam records...\n")
    
    predictions = []
    for i, exam in enumerate(test_exams, 1):
        # Prepare features
        test_features = {k: v for k, v in exam.items() if k not in ['_id', 'userID', 'createdDate']}
        if 'average_time' in test_features:
            test_features['avg_time_per_question'] = test_features.pop('average_time')
        
        # Predict
        pred_result = predict_expertise_level(test_features)
        predictions.append(pred_result)
        
        print(f"Sample {i}:")
        print(f"  Accuracy: {test_features.get('accuracy', 0)*100:.1f}%")
        print(f"  Predicted Level: {pred_result.get('expertise_level', 'N/A')}")
        print(f"  Confidence: {pred_result.get('confidence', 0):.2%}")
    
    print(f"\n✓ Completed {len(predictions)} predictions")
else:
    print("⚠ No exam data available for testing")

# Test 3: Model Performance Metrics
print("\n[TEST 3] Model Performance Summary")
print("-" * 60)

if 'model' in globals() and model is not None:
    print("Training Metrics:")
    print(f"  F1 Score (Macro): {f1:.4f}")
    print(f"  Test Set Size: {len(y_test)}")
    print(f"  Training Set Size: {len(y_train)}")
    
    # Feature importance (top 5)
    if hasattr(model, 'get_feature_importance'):
        importance = model.get_feature_importance()
        feature_names = FEATURES
        feature_importance = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)
        
        print(f"\n  Top 5 Important Features:")
        for feat, imp in feature_importance[:5]:
            print(f"    {feat}: {imp:.2f}")
else:
    print("⚠ Model not trained in this session")

# Test 4: Validation with Test Set
print("\n[TEST 4] Test Set Validation")
print("-" * 60)

if 'X_test' in globals() and 'y_test' in globals():
    # Make predictions on test set
    test_predictions = model.predict(X_test)
    
    # Calculate accuracy per class
    from sklearn.metrics import accuracy_score
    
    overall_accuracy = accuracy_score(y_test, test_predictions)
    print(f"Overall Test Accuracy: {overall_accuracy:.2%}")
    
    # Per-class accuracy
    print(f"\nPer-Class Performance:")
    for cls in np.unique(y_test):
        mask = y_test == cls
        if mask.sum() > 0:
            cls_accuracy = accuracy_score(y_test[mask], test_predictions[mask])
            cls_name = label_encoder.inverse_transform([cls])[0]
            print(f"  Level {cls_name}: {cls_accuracy:.2%} ({mask.sum()} samples)")
    
    print(f"\n✓ Test set validation complete")
else:
    print("⚠ Test set not available")

# Test 5: Edge Case Testing
print("\n[TEST 5] Edge Case Testing")
print("-" * 60)

edge_cases = [
    {
        "name": "Perfect Score",
        "features": {
            "total_questions": 40,
            "correct_answers": 40,
            "accuracy": 1.0,
            "easy_accuracy": 1.0,
            "medium_accuracy": 1.0,
            "hard_accuracy": 1.0,
            "avg_time_per_question": 5.0
        }
    },
    {
        "name": "Zero Score",
        "features": {
            "total_questions": 40,
            "correct_answers": 0,
            "accuracy": 0.0,
            "easy_accuracy": 0.0,
            "medium_accuracy": 0.0,
            "hard_accuracy": 0.0,
            "avg_time_per_question": 10.0
        }
    },
    {
        "name": "Intermediate Performance",
        "features": {
            "total_questions": 40,
            "correct_answers": 25,
            "accuracy": 0.625,
            "easy_accuracy": 0.8,
            "medium_accuracy": 0.6,
            "hard_accuracy": 0.45,
            "avg_time_per_question": 7.5
        }
    }
]

for case in edge_cases:
    pred = predict_expertise_level(case["features"])
    print(f"\n{case['name']}:")
    print(f"  Predicted Level: {pred.get('expertise_level', 'N/A')}")
    print(f"  Confidence: {pred.get('confidence', 0):.2%}")

print("\n" + "="*60)
print("✓ ALL TESTS COMPLETED")
print("="*60)
